In [1]:
from LSTM import LSTM, Layer_Dense
import numpy as np

np.random.seed(0)

# ----------------------------
# Toy "machine translation" data:
# map sequences of length L over vocab {1..V-2} to reversed sequence
# with <SOS>=0, <EOS>=V-1
# ----------------------------

V = 8             # vocab size
SOS = 0
EOS = V - 1
max_len = 5
n_samples = 1000

In [3]:
def generate_sample():
    length = np.random.randint(1, max_len+1)
    # tokens in 1..V-2
    src_seq = np.random.randint(1, V-1, size=length)
    # target is reversed source
    tgt_seq = src_seq[::-1]
    # add EOS
    tgt_seq = np.concatenate([tgt_seq, [EOS]])
    return src_seq, tgt_seq

In [4]:
# generate_sample()

In [5]:
# generate_sample()

In [6]:
def one_hot_sequence(seq, T):
    # seq: array of token ids (int)
    # T: fixed time length (pad with EOS)
    x = np.zeros((T, V))
    for t in range(len(seq)):
        x[t, seq[t]] = 1.0
    for t in range(len(seq), T):
        x[t, EOS] = 1.0
    return x

In [75]:
T_enc = max_len
src, tgt = generate_sample()

# encoder input: src, padded
# X_enc.append(one_hot_sequence(src, T_enc))
print(src)
one_hot_sequence(src, T_enc)

[6 1 4 4 4]


array([[0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.]])

In [76]:
src, tgt

(array([6, 1, 4, 4, 4], dtype=int32), array([4, 4, 4, 1, 6, 7]))

In [77]:
dec_in = np.concatenate([[SOS], tgt[:-1]])
dec_in

array([0, 4, 4, 4, 1, 6])

In [7]:
# build dataset
T_enc = max_len
T_dec = max_len + 1  # reversed plus EOS
X_enc = []
X_dec_in = []
Y_dec = []

for _ in range(n_samples):
    src, tgt = generate_sample()

    # encoder input: src, padded
    X_enc.append(one_hot_sequence(src, T_enc))

    # decoder input: [SOS] + tgt[:-1]
    dec_in = np.concatenate([[SOS], tgt[:-1]])
    X_dec_in.append(one_hot_sequence(dec_in, T_dec))

    # decoder output (targets): tgt, padded to T_dec
    y = np.full(T_dec, EOS, dtype=int)
    y[:len(tgt)] = tgt
    Y_dec.append(y)

In [79]:
Y_dec[0]

array([6, 4, 7, 7, 7, 7])

In [80]:
X_dec_in[0]

array([[1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.]])

In [8]:
X_enc = np.array(X_enc)     # (N, T_enc, V)
X_dec_in = np.array(X_dec_in) # (N, T_dec, V)
Y_dec = np.array(Y_dec)     # (N, T_dec)

# ----------------------------
# Model: encoder LSTM + decoder LSTM + output dense
# ----------------------------

In [9]:
n_neurons = 64

encoder = LSTM(n_neurons, input_dim=V)
decoder = LSTM(n_neurons, input_dim=V)

In [83]:
encoder.dUf.fill(0.0)

#### Decoder output: from h_t^dec (size n_neurons) -> vocab logits (size V)
#### We'll operate per time-step, so we can have a single dense over n_neurons -> V
#### but we can also flatten time dimension; for clarity we'll process all times at once.

In [10]:
hidden_size = encoder.n_neurons
output_layer = Layer_Dense(hidden_size, V)
# output_layer = Layer_Dense(n_neurons, V)

lr = 1e-2
n_epochs = 50
batch_size = 32

In [11]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(x)
    return exp / np.sum(exp, axis=-1, keepdims=True)

In [86]:
X_enc.shape

(1000, 5, 8)

In [87]:
type(X_enc)

numpy.ndarray

In [88]:
perm = np.random.permutation(n_samples)
perm

array([654, 762, 370, 416, 990, 346, 157, 426,  63, 640, 315, 110, 141,
       403, 259, 765, 631, 538, 815, 524, 617, 116, 759, 488, 914, 218,
       603, 625, 940, 689, 987,  40,  28, 392, 741, 781, 730, 769, 874,
       536, 415, 999, 878, 499, 852, 511, 496,   2,  61, 473, 143, 837,
       585,  80, 672, 271, 112, 319, 304, 341, 798, 183, 222, 841, 943,
       456, 272, 118, 703, 217, 417, 237, 360, 197, 475, 936, 220, 899,
       281, 549, 550, 707, 858, 584, 337, 954, 158, 334, 514, 686, 205,
       431, 962, 268, 207, 834, 434, 910, 604, 895, 100, 301, 682, 907,
       606, 870, 343, 352, 576, 134, 885,  45, 386, 708, 850, 593,   5,
       994, 288, 375,  32, 425,  90, 868, 778, 487, 745, 634, 853, 839,
       960, 129, 893,  29, 670,  78, 575, 746, 748, 764,  16, 777,  57,
       454, 150, 424,  65, 124,  69, 154, 829, 636, 709, 955, 338,  92,
        44, 269, 698, 242, 944, 732, 512,  48, 361, 292, 842, 283, 831,
       804, 615, 450, 711, 303, 979, 381, 572, 809, 401, 626, 19

In [89]:
X_enc = X_enc[perm]

In [90]:
type(X_enc), X_enc.shape

(numpy.ndarray, (1000, 5, 8))

In [12]:
for epoch in range(n_epochs):
    perm = np.random.permutation(n_samples)
    X_enc = X_enc[perm]
    X_dec_in = X_dec_in[perm]
    Y_dec = Y_dec[perm]

    total_loss = 0.0

    for start in range(0, n_samples, batch_size):
        end = start + batch_size
        if end > n_samples:
            break

        x_e = X_enc[start:end]      # (B, T_enc, V)
        x_d = X_dec_in[start:end]   # (B, T_dec, V)
        y   = Y_dec[start:end]      # (B, T_dec)

        B = x_e.shape[0]

        # Accumulate grads manually across batch (since your LSTM isn't batched)
        # We'll loop over samples in the batch; it's slow but conceptually simple.

        batch_loss = 0.0

        # Zero parameter grads
        # (They will be accumulated across samples and then updated once)
        # Encoder
        encoder.dUf.fill(0.0)
        encoder.dUi.fill(0.0); encoder.dUo.fill(0.0); encoder.dUg.fill(0.0)
        encoder.dWf.fill(0.0); encoder.dWi.fill(0.0); encoder.dWo.fill(0.0); encoder.dWg.fill(0.0)
        encoder.dbf.fill(0.0); encoder.dbi.fill(0.0); encoder.dbo.fill(0.0); encoder.dbg.fill(0.0)

        # Decoder
        decoder.dUf.fill(0.0); decoder.dUi.fill(0.0); decoder.dUo.fill(0.0); decoder.dUg.fill(0.0)
        decoder.dWf.fill(0.0); decoder.dWi.fill(0.0); decoder.dWo.fill(0.0); decoder.dWg.fill(0.0)
        decoder.dbf.fill(0.0); decoder.dbi.fill(0.0); decoder.dbo.fill(0.0); decoder.dbg.fill(0.0)

        # Output layer
        output_layer.dweights = np.zeros_like(output_layer.weights)
        output_layer.dbiases = np.zeros_like(output_layer.biases)

        for b in range(B):
            # -------- Encoder forward --------
            # shape (T_enc, V) -> list of (V,1) per time
            enc_in = x_e[b]        # (T_enc, V)
            encoder.forward(enc_in)  # h0=c0=0

            # final encoder states
            h_enc = encoder.H[encoder.T]   # (n_neurons, 1)
            c_enc = encoder.C[encoder.T]   # (n_neurons, 1)

            # -------- Decoder forward --------
            dec_in = x_d[b]        # (T_dec, V)
            decoder.forward(dec_in, h0=h_enc, c0=c_enc)

            # decoder hidden states H[1:] shape: list of (n_neurons,1)
            H_dec = np.array(decoder.H[1:])  # (T_dec, n_neurons, 1)
            H_dec = H_dec.reshape(decoder.T, n_neurons)  # (T_dec, n_neurons)

            # output layer forward: map each h_t -> logits over vocab
            output_layer.forward(H_dec)  # (T_dec, V)

            logits = output_layer.output
            probs = softmax(logits)      # (T_dec, V)

            # Cross-entropy loss
            # y[b,t] is integer class
            # L = -sum_t log p_t[y_t]
            idx = y[b]                   # (T_dec,)

            # print("decoder.T =", decoder.T)
            # print("len(y[b]) =", len(y[b]))
            logp = -np.log(probs[np.arange(decoder.T), idx] + 1e-12)
            loss = np.mean(logp)
            batch_loss += loss

            # -------- Backward pass --------

            # dL/dlogits for softmax + cross-entropy
            dlogits = probs.copy()
            dlogits[np.arange(decoder.T), idx] -= 1.0
            dlogits /= decoder.T  # mean over time

            # backprop through output dense
            output_layer.backward(dlogits)  # sets dweights, dbiases, dinputs

            # dvalues for decoder LSTM: dL/dh_t
            dH_dec = output_layer.dinputs  # (T_dec, n_neurons)
            decoder.backward(dH_dec)

            # For encoder: gradient only via decoder initial state
            # decoder.backward has updated decoder.dct and dht at t=0 into internal states;
            # but your LSTM.backward does not expose dH[0] or dC[0].
            # For a simple approximate link, we can treat decoder.H[0] = encoder.H[T_enc]
            # and ignore c-gradients (not ideal, but consistent with your API).
            # To do it correctly, LSTM.backward would need to store dH[0].

            # Here we approximate: encoder receives decoder's first hidden gradient
            dh_enc_T = decoder.Wf.T @ decoder.Sigmf[0].dinputs + \
                       decoder.Wi.T @ decoder.Sigmi[0].dinputs + \
                       decoder.Wo.T @ decoder.Sigmo[0].dinputs + \
                       decoder.Wg.T @ decoder.Tanh1[0].dinputs

            dH_enc = np.zeros((encoder.T, encoder.n_neurons))
            dH_enc[-1] = dh_enc_T.reshape(-1)

            encoder.backward(dH_enc)

            # Accumulate grads into global param grads (already being accumulated inside encoder/decoder.backward)

        # average loss over batch
        batch_loss /= B
        total_loss += batch_loss

        # -------- SGD update after batch --------
        # encoder update
        encoder.Uf -= lr * encoder.dUf
        encoder.Ui -= lr * encoder.dUi
        encoder.Uo -= lr * encoder.dUo
        encoder.Ug -= lr * encoder.dUg

        encoder.Wf -= lr * encoder.dWf
        encoder.Wi -= lr * encoder.dWi
        encoder.Wo -= lr * encoder.dWo
        encoder.Wg -= lr * encoder.dWg

        encoder.bf -= lr * encoder.dbf
        encoder.bi -= lr * encoder.dbi
        encoder.bo -= lr * encoder.dbo
        encoder.bg -= lr * encoder.dbg

        # decoder update
        decoder.Uf -= lr * decoder.dUf
        decoder.Ui -= lr * decoder.dUi
        decoder.Uo -= lr * decoder.dUo
        decoder.Ug -= lr * decoder.dUg

        decoder.Wf -= lr * decoder.dWf
        decoder.Wi -= lr * decoder.dWi
        decoder.Wo -= lr * decoder.dWo
        decoder.Wg -= lr * decoder.dWg

        decoder.bf -= lr * decoder.dbf
        decoder.bi -= lr * decoder.dbi
        decoder.bo -= lr * decoder.dbo
        decoder.bg -= lr * decoder.dbg

        # output dense layer update
        output_layer.weights -= lr * output_layer.dweights
        output_layer.biases  -= lr * output_layer.dbiases

    print(f"Epoch {epoch+1}/{n_epochs}, loss: {total_loss:.4f}")

Epoch 1/50, loss: 46.9893
Epoch 2/50, loss: 39.8694
Epoch 3/50, loss: 38.2326
Epoch 4/50, loss: 37.6104
Epoch 5/50, loss: 37.1724
Epoch 6/50, loss: 36.6518
Epoch 7/50, loss: 36.1145
Epoch 8/50, loss: 35.5327
Epoch 9/50, loss: 35.0428
Epoch 10/50, loss: 34.3658
Epoch 11/50, loss: 34.0456
Epoch 12/50, loss: 33.7434
Epoch 13/50, loss: 33.2990
Epoch 14/50, loss: 33.0182
Epoch 15/50, loss: 32.8261
Epoch 16/50, loss: 32.5510
Epoch 17/50, loss: 32.2080
Epoch 18/50, loss: 32.0091
Epoch 19/50, loss: 31.8791
Epoch 20/50, loss: 31.6286
Epoch 21/50, loss: 31.4976
Epoch 22/50, loss: 31.4207
Epoch 23/50, loss: 31.1938
Epoch 24/50, loss: 31.0758
Epoch 25/50, loss: 30.9591
Epoch 26/50, loss: 30.9201
Epoch 27/50, loss: 30.7104
Epoch 28/50, loss: 30.5593
Epoch 29/50, loss: 30.4278
Epoch 30/50, loss: 30.3520
Epoch 31/50, loss: 30.2898
Epoch 32/50, loss: 30.1287
Epoch 33/50, loss: 30.1137
Epoch 34/50, loss: 30.0886
Epoch 35/50, loss: 29.9489
Epoch 36/50, loss: 29.8502
Epoch 37/50, loss: 29.8475
Epoch 38/5

## Test Mode

In [13]:
def one_hot_token(token, V):
    x = np.zeros((1, V))
    x[0, token] = 1.0
    return x

In [14]:
def encode_sequence(encoder, X_enc):
    h, c = encoder.forward(X_enc)
    return h[-1], c[-1]   # final state

In [15]:
def decode_sequence(decoder, output_layer, h0, c0, V, SOS, EOS, max_len):

    tokens = [SOS]

    for t in range(max_len):

        # build decoder input sequence so far
        x = np.zeros((len(tokens), V))
        for i, tok in enumerate(tokens):
            x[i, tok] = 1.0

        # run decoder on whole sequence
        h_seq, c_seq = decoder.forward(x, h0, c0)

        h = h_seq[-1]

        logits = output_layer.forward(h.T)
        probs = softmax(logits[0])

        token = np.argmax(probs)

        if token == EOS:
            break

        tokens.append(token)

    return tokens[1:]


In [16]:
def predict(src_seq, encoder, decoder, output_layer, V, SOS, EOS, T_enc, T_dec):
    # pad source
    X_enc = one_hot_sequence(src_seq, T_enc)

    # encode
    h0, c0 = encode_sequence(encoder, X_enc)

    # decode
    pred = decode_sequence(decoder, output_layer, h0, c0,
                           V, SOS, EOS, T_dec)

    return pred

In [17]:
for _ in range(5):
    src, tgt = generate_sample()
    pred = predict(src, encoder, decoder, output_layer,
                   V, SOS, EOS, T_enc, T_dec)

    print("SRC :", src)
    print("TGT :", tgt[:-1])       # without EOS
    print("PRED:", pred)
    print("-----")

SRC : [5]
TGT : [5]
PRED: [np.int64(6)]
-----
SRC : [4 3 2 4]
TGT : [4 2 3 4]
PRED: [np.int64(2), np.int64(2), np.int64(2), np.int64(1)]
-----
SRC : [4]
TGT : [4]
PRED: [np.int64(6)]
-----
SRC : [6 4 1 4]
TGT : [4 1 4 6]
PRED: [np.int64(2), np.int64(2), np.int64(2)]
-----
SRC : [1 4]
TGT : [4 1]
PRED: [np.int64(2)]
-----
